# Chapter 10 — Treeview tables: columns, sorting, selection, and large result sets

**Book role:** Core GUI  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Present tabular records with ttk.Treeview.
- Define headings, column widths, and alignment explicitly.
- Implement stable sorting and row selection.
- Plan pagination or virtualization for datasets larger than a practical widget view.

## 1. Treeview’s data model

`ttk.Treeview` displays hierarchical items and columns. For a flat data table, the tree column (`#0`) is often hidden and named columns represent the DataFrame’s selected fields. Define `columns`, `show="headings"`, column heading text, width, minimum width, and anchor. Use a vertical scrollbar and, for wide records, a horizontal scrollbar.

Treeview stores display values, not a pandas DataFrame. Keep the canonical DataFrame or record model separate. A reliable table layer maps a displayed item ID to a record identifier or row index, rather than assuming the display order is the source order. Rebuilding the entire table on every keystroke is expensive; debounce search and update only when needed.

### Worked example 1: Display DataFrame rows in Treeview

The example is appropriate for a tiny demonstration. A real view should format dates and currency, handle nulls, store stable row identifiers, and cap how many records are inserted at once.

In [ ]:
import tkinter as tk
from tkinter import ttk
import pandas as pd

sample = pd.DataFrame({
    "order_id": [101, 102, 103],
    "region": ["North", "West", "South"],
    "revenue": [1250.50, 980.00, 2030.75],
})

root = tk.Tk()
root.title("Records")
root.geometry("600x260")
columns = list(sample.columns)
table = ttk.Treeview(root, columns=columns, show="headings", height=8)
for column in columns:
    table.heading(column, text=column.replace("_", " ").title())
    table.column(column, width=150, anchor="w")
scrollbar = ttk.Scrollbar(root, orient="vertical", command=table.yview)
table.configure(yscrollcommand=scrollbar.set)
table.pack(side="left", fill="both", expand=True, padx=(8, 0), pady=8)
scrollbar.pack(side="right", fill="y", padx=8, pady=8)
for record in sample.itertuples(index=False, name=None):
    table.insert("", "end", values=record)
root.mainloop()

## 2. Sort values, not formatted strings

A simplistic sort can compare the visible text. That fails when numeric values have commas, currency symbols, or different number formats: lexicographic order can place “100” before “20.” Sort using the underlying Python values, with an explicit missing-value policy. For stable sorting, preserve original order among equal keys. For large datasets, sort the DataFrame or database result and redraw a bounded page.

Treeview item IDs are strings; they are not automatically domain primary keys. Store a stable `record_id` mapping or use the source primary key as the item identifier when uniqueness and character rules permit. When a selected record is edited or deleted, resolve it through the model/repository, then refresh the appropriate rows.

### Worked example 2: Sort a numeric DataFrame column correctly

Keep raw numeric types for computation and sorting. Add formatting only at the last presentation boundary.

In [ ]:
import pandas as pd

frame = pd.DataFrame({"product": ["A", "B", "C"], "revenue": [100, 20, 3]})
# Wrong conceptual approach: sorting the display strings gives lexical order.
frame["revenue_display"] = frame["revenue"].map(lambda value: f"{value:,}")
print("Underlying numeric sort:")
print(frame.sort_values("revenue", ascending=True)[["product", "revenue"]])
print("Display strings are for presentation only:")
print(frame[["product", "revenue_display"]])

## 3. Pagination and the data-to-view boundary

A widget is not a replacement for a database query engine. Showing 100,000 rows as Treeview items is usually a poor interaction pattern even if the machine has enough memory. Use server-side filters and pagination for database-backed data, or a bounded view for an in-memory DataFrame. Report the total number of matches and which page is visible.

Keep three counts distinct: raw records loaded, rows matching current filters, and rows displayed in the current page. This prevents users from assuming the table is the whole dataset. When exporting, document whether the export means “current page,” “all filtered rows,” or “all source rows.” A deliberate label and confirmation prevent accidental partial exports.

## Guided lab: Lab: searchable record explorer

Build a table that reads the sample sales CSV, displays at most 200 rows, supports text filtering, numeric sorting, row selection, and a visible match count. Add a clear indication when only the first page is displayed. Exercise empty data, a missing column, null values, and a file containing more rows than the display limit.

## Review questions and practice

1. Why should a table preserve a mapping to stable record identifiers?
2. Why is sorting display strings unreliable for numeric values?
3. What is the difference between total matches and displayed rows?
4. Why should an export action state exactly which records it includes?
5. What should happen when the current filter returns zero rows?

## Further reading

- Ttk Treeview: https://docs.python.org/3/library/tkinter.ttk.html#treeview
- pandas sorting: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.